In [ ]:
import os
from pymongo import MongoClient

MONGO_DB_URL = os.environ["MONGO_DB_URL"]

print("Connecting to MongoDB...")
client = MongoClient(MONGO_DB_URL)

print(client.list_database_names())

In [ ]:
db = client["TrainDelaysDB"]
# Show collection names
print("Collections in TrainDelaysDB:")
print(db.list_collection_names())

In [ ]:
# show basic info about the collection
collection = db["trainStateSnaphots"]
print(f"Collection 'trainStateSnaphots' stats:")
print(db.command("collstats", "trainStateSnaphots"))
sample_doc = collection.find_one()
print("Sample document from 'trainStateSnaphots':")
print(sample_doc)

In [ ]:
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
from tqdm import tqdm

file_name = '../data/raw/train_data.parquet'
batch_size = 1000 

if os.path.exists(file_name):
    os.remove(file_name)

total_docs = collection.estimated_document_count()
cursor = collection.find({})
writer = None
batch = []

try:
    with tqdm(total=total_docs, unit="docs") as pbar:
        for doc in cursor:
            # 1. Stringify the ID
            if '_id' in doc:
                doc['_id'] = str(doc['_id'])
            
            # 2. JSON-encode the 'trains' list to bypass nested schema errors
            if 'trains' in doc:
                doc['trains'] = json.dumps(doc['trains'], default=str)
            else:
                doc['trains'] = "[]"
            
            batch.append(doc)
            
            if len(batch) >= batch_size:
                df_batch = pd.DataFrame(batch)
                
                # Align columns to the very first batch
                if writer is None:
                    master_columns = df_batch.columns.tolist()
                else:
                    for col in master_columns:
                        if col not in df_batch.columns:
                            df_batch[col] = None
                    df_batch = df_batch[master_columns]

                # Convert to Arrow Table
                table = pa.Table.from_pandas(df_batch)
                
                if writer is None:
                    writer = pq.ParquetWriter(file_name, table.schema)
                
                writer.write_table(table)
                batch = []
                pbar.update(batch_size)

        # Final batch
        if batch:
            df_batch = pd.DataFrame(batch)
            for col in master_columns:
                if col not in df_batch.columns:
                    df_batch[col] = None
            df_batch = df_batch[master_columns]
            writer.write_table(pa.Table.from_pandas(df_batch, schema=writer.schema))
            pbar.update(len(batch))

finally:
    if writer:
        writer.close()

print(f"\nSaved! To read back: df['trains'] = df['trains'].apply(json.loads)")
